In [ ]:
import argparse
import json
import os
import random
from dataclasses import dataclass
from typing import Any

import torch
from PIL import Image
from torch.utils.data import Dataset
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    TrainerCallback,
    Trainer,
    TrainingArguments,
)

try:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    PEFT_AVAILABLE = True
except ImportError:
    PEFT_AVAILABLE = False


# ---------------------------------------------------------------------------
# Instruction prompts — randomly sampled per example
# ---------------------------------------------------------------------------

INSTRUCTIONS = [
    "What is written in the image?",
    "Transcribe the text shown in this image.",
    "Read the image and output the text.",
    "Extract all text from this image.",
    "Please OCR this image.",
    "What text appears in the image?",
    "Read out the contents of this image.",
    "Perform OCR on the image and return the text.",
    "Give me the text content of this image.",
    "What does this image say?",
]


# ---------------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------------
import random

class BrailleOCRDataset(Dataset):
    def __init__(
        self,
        image_dir: str = "datav2/train/images",
        split: str = "train",
        seed: int = 42,
        label_path: str = "datav2/train/labels/labels.jsonl",
    ):

        if split == "val":
            image_dir = image_dir.replace("train", "val")
            label_path = label_path.replace("train", "val")
        
        label_path = os.path.abspath(label_path)

        print(f"[Dataset] Loading {split} split from {label_path}")
        try:
            all_records = []
            with open(label_path, "r") as f:
                for line in f:
                    all_records.append(json.loads(line))
        
        except:
            all_records = json.load(open(label_path, "r"))

        print(f"[Dataset] Loaded {len(all_records)} total records from {label_path}")

        self.records = all_records
        self.image_dir = image_dir
        self.instructions = INSTRUCTIONS

        print(f"[Dataset] split={split}  size={len(self.records)}")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]

        # Load image
        img_path = os.path.join(self.image_dir, record["image"]) if "id" in record else os.path.join(self.image_dir, record["file"])
        image = Image.open(img_path).convert("RGB")
    
        # Select random instruction
        instruction = random.choice(self.instructions)

        # Get target text
        answer = record["reasoning_text"] if "reasoning_text" in record else record["source_text"]
        return {
            "image": image,
            "instruction": instruction,
            "answer": answer,
        }


# ---------------------------------------------------------------------------
# Collator — builds chat-formatted inputs and masks the prompt tokens
# ---------------------------------------------------------------------------
@dataclass
class ChandraCollator:
    processor: Any

    def __call__(self, features):
        images = [f["image"] for f in features]

        # Build the full chat (user + assistant) for each example
        full_texts = []
        prompt_texts = []
        for f in features:
            messages_full = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image"},
                        {"type": "text", "text": f["instruction"]},
                    ],
                },
                {
                    "role": "assistant",
                    "content": [{"type": "text", "text": f["answer"] + self.processor.tokenizer.eos_token}],
                },
            ]
            messages_prompt = [messages_full[0]]

            full_text = self.processor.apply_chat_template(
                messages_full, tokenize=False, add_generation_prompt=False
            )
            prompt_text = self.processor.apply_chat_template(
                messages_prompt, tokenize=False, add_generation_prompt=True
            )
            full_texts.append(full_text)
            prompt_texts.append(prompt_text)

        # Tokenize + process images
        batch = self.processor(
            text=full_texts,
            images=images,
            return_tensors="pt",
            padding=True,
            eos_token_id=self.processor.tokenizer.eos_token_id,
        )

        # Build labels — mask everything that is part of the prompt (and padding)
        input_ids = batch["input_ids"]
        labels = input_ids.clone()

        pad_id = self.processor.tokenizer.pad_token_id
        eos_id = self.processor.tokenizer.eos_token_id
        if pad_id is not None:
            labels[labels == pad_id] = -100

        # Mask image tokens if present
        image_token_id = getattr(self.processor.tokenizer, "image_token_id", None)
        if image_token_id is None:
            image_token = getattr(self.processor.tokenizer, "image_token", None)
            if image_token is not None:
                image_token_id = self.processor.tokenizer.convert_tokens_to_ids(image_token)
        if image_token_id is not None and image_token_id >= 0:
            labels[labels == image_token_id] = -100

        # Mask prompt portion per example
        for i, prompt_text in enumerate(prompt_texts):
            prompt_ids = self.processor.tokenizer(
                prompt_text, add_special_tokens=False, return_tensors="pt"
            ).input_ids[0]
            prompt_len = prompt_ids.shape[0]
            # With left-padding the prompt sits after the pad tokens, so we
            # find where real content starts and mask prompt_len tokens from there
            seq = input_ids[i]
            if pad_id is not None:
                non_pad = (seq != pad_id).nonzero(as_tuple=True)[0]
                start = non_pad[0].item() if len(non_pad) > 0 else 0
            else:
                start = 0
            labels[i, start : start + prompt_len] = -100
        
        batch["labels"] = labels
        # print(batch["input_ids"][0][-10:])
        # print(batch["labels"][0][-10:])
        return batch


# ---------------------------------------------------------------------------
# Sample prediction callback during training
# ---------------------------------------------------------------------------
class SamplePredictionCallback(TrainerCallback):
    def __init__(
        self,
        processor: Any,
        dataset: Dataset,
        every_steps: int = 100,
        num_samples: int = 1,
        max_new_tokens: int = 128,
        seed: int = 42,
        output_dir: str = "./sample_predictions",  # <-- new
    ):
        self.processor = processor
        self.dataset = dataset
        self.every_steps = max(1, every_steps)
        self.num_samples = max(1, num_samples)
        self.max_new_tokens = max(1, max_new_tokens)
        self.rng = random.Random(seed)
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        self.all_results = []  # accumulates across all steps

    def _generate_prediction(self, model, image: Image.Image, instruction: str, device: torch.device) -> str:
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": instruction},
                ],
            },
        ]
        prompt = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.processor(text=[prompt], images=[[image]], return_tensors="pt", padding=True)
        inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
            )

        prompt_len = inputs["input_ids"].shape[1]
        generated = output_ids[0, prompt_len:]
        return self.processor.tokenizer.decode(generated, skip_special_tokens=True).strip()

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step <= 0 or state.global_step % self.every_steps != 0:
            return control

        model = kwargs.get("model")
        if model is None or len(self.dataset) == 0:
            return control

        device = next(model.parameters()).device
        was_training = model.training
        model.eval()

        print("\n" + "=" * 80)
        print(f"[Sample Eval] step={state.global_step}")
        print("=" * 80)

        step_results = []
        for i in range(self.num_samples):
            idx = self.rng.randrange(len(self.dataset))
            sample = self.dataset[idx]
            image = sample["image"]
            instruction = sample["instruction"]
            reference = sample["answer"]

            try:
                prediction = self._generate_prediction(model, image, instruction, device)
            except Exception as e:
                print(f"[Sample Eval] sample={i+1} failed: {e}")
                continue

            exact_match = prediction.strip() == reference.strip()
            print(f"Sample {i+1}/{self.num_samples}")
            print(f"Instruction: {instruction}")
            print(f"Reference  : {reference[:160]}")
            print(f"Prediction : {prediction[:160]}")
            print(f"Exact Match: {exact_match}")
            print("-" * 80)

            step_results.append({
                "step": state.global_step,
                "sample_idx": idx,
                "instruction": instruction,
                "reference": reference,
                "prediction": prediction,
                "exact_match": exact_match,
            })

        # Save per-step file
        if step_results:
            step_file = os.path.join(self.output_dir, f"step_{state.global_step:07d}.json")
            with open(step_file, "w", encoding="utf-8") as f:
                json.dump(step_results, f, ensure_ascii=False, indent=2)

            # Also maintain a running cumulative file
            self.all_results.extend(step_results)
            all_file = os.path.join(self.output_dir, "all_predictions.json")
            with open(all_file, "w", encoding="utf-8") as f:
                json.dump(self.all_results, f, ensure_ascii=False, indent=2)

            print(f"[Sample Eval] Saved {len(step_results)} predictions to {step_file}")

        if was_training:
            model.train()

        return control


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--image_dir", required=True, help="Root dir that image_path is relative to")
    parser.add_argument("--label_path", default="data/labels/labels.jsonl")
    parser.add_argument("--model_id", default="datalab-to/chandra-ocr-2")
    parser.add_argument("--output_dir", default="./chandra-braille-ft")

    parser.add_argument("--epochs", type=int, default=1)
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--grad_accum", type=int, default=1)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--warmup_ratio", type=float, default=0.03)
    parser.add_argument("--weight_decay", type=float, default=0.0)
    parser.add_argument("--max_grad_norm", type=float, default=1.0)
    parser.add_argument("--seed", type=int, default=42)

    parser.add_argument("--use_lora", action="store_true", help="Fine-tune with LoRA adapters (recommended)")
    parser.add_argument("--lora_r", type=int, default=16)
    parser.add_argument("--lora_alpha", type=int, default=32)
    parser.add_argument("--lora_dropout", type=float, default=0.05)

    parser.add_argument("--bf16", action="store_true", default=True)
    parser.add_argument("--gradient_checkpointing", action="store_true", default=False)

    parser.add_argument("--eval_split", default="val")
    parser.add_argument("--no_eval", action="store_true")
    parser.add_argument("--sample_eval_steps", type=int, default=100, help="Run sample prediction eval every N steps")
    parser.add_argument("--sample_eval_count", type=int, default=5, help="Number of random samples per sample-eval run")
    parser.add_argument("--sample_eval_max_new_tokens", type=int, default=128, help="Max tokens for sample prediction generation")
    parser.add_argument("--disable_sample_eval", action="store_true", help="Disable periodic sample prediction during training")
    parser.add_argument("--label_file", type=str, default="labels_with_reasoning.json", help="Label file to use for training")
    args = parser.parse_args()

    random.seed(args.seed)
    torch.manual_seed(args.seed)

    # -----------------------------------------------------------------------
    # Processor + model
    # -----------------------------------------------------------------------
    print(f"[Load] processor + model from {args.model_id}")
    processor = AutoProcessor.from_pretrained(args.model_id)
    processor.tokenizer.padding_side = "right"  # right-pad for training
    processor.tokenizer.eos_token = "[EOS]"  # use pad token as EOS for generation

    model = AutoModelForImageTextToText.from_pretrained(
        args.model_id,
        dtype=torch.bfloat16 if args.bf16 else torch.float32,
        attn_implementation="sdpa",
    )
    model.config.use_cache = False

    if args.gradient_checkpointing:
        model.gradient_checkpointing_enable()
        if hasattr(model, "enable_input_require_grads"):
            model.enable_input_require_grads()

    # LoRA
    if args.use_lora:
        if not PEFT_AVAILABLE:
            raise RuntimeError("peft is not installed. Run: pip install peft")
        print("[LoRA] wrapping model with LoRA adapters")
        lora_config = LoraConfig(
            r=args.lora_r,
            lora_alpha=args.lora_alpha,
            lora_dropout=args.lora_dropout,
            bias="none",
            task_type="CAUSAL_LM",
            # Common attention projection names — peft will match whichever exist
            target_modules=[
                "q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj",
            ],
        )
        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()

    # -----------------------------------------------------------------------
    # Datasets
    # -----------------------------------------------------------------------
    train_ds = BrailleOCRDataset(
        image_dir=args.image_dir,
        split="train",
        label_path=args.label_path,
    )
    eval_ds = None

    print(train_ds[0])

    if not args.no_eval:
        try:
            eval_ds = BrailleOCRDataset(
                image_dir=args.image_dir,
                split=args.eval_split,
                label_path=args.label_path,
            )
            if len(eval_ds) == 0:
                eval_ds = None
        except Exception as e:
            print(f"[Eval] disabled: {e}")
            eval_ds = None

    from torch.utils.data import Subset

    if eval_ds is not None and len(eval_ds) > 100:
        eval_indices = random.sample(range(len(eval_ds)), 100)
        sample_eval_dataset = Subset(eval_ds, eval_indices)
        print(f"[Eval] Subsampled to {len(sample_eval_dataset)} samples")

    collator = ChandraCollator(processor=processor)

    # -----------------------------------------------------------------------
    # TrainingArguments
    # -----------------------------------------------------------------------
    training_args = TrainingArguments(
        output_dir=args.output_dir,
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.lr,
        warmup_ratio=args.warmup_ratio,
        weight_decay=args.weight_decay,
        max_grad_norm=args.max_grad_norm,
        logging_steps=100,
        save_strategy="steps",
        save_total_limit=2,
        eval_strategy="steps" if eval_ds is not None else "no",
        bf16=args.bf16,
        gradient_checkpointing=args.gradient_checkpointing,
        remove_unused_columns=False,
        dataloader_num_workers=2,
        report_to="none",
        seed=args.seed,
        label_names=["labels"],
        load_best_model_at_end = True if eval_ds is not None else False,
        metric_for_best_model = "loss"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=collator,
    )

    if not args.disable_sample_eval:
        sample_eval_dataset = eval_ds if eval_ds is not None else train_ds
        if eval_ds is not None:
            print("Adding Sample Prediction Callback on eval_ds")

        trainer.add_callback(
            SamplePredictionCallback(
                processor=processor,
                dataset=sample_eval_dataset,
                every_steps=args.sample_eval_steps,
                num_samples=args.sample_eval_count,
                max_new_tokens=args.sample_eval_max_new_tokens,
                seed=args.seed,
            )
        )

    # -----------------------------------------------------------------------
    # Train
    # -----------------------------------------------------------------------
    trainer.train(
        resume_from_checkpoint="output/chandra-braille-ft-new-2/checkpoint-2000" if os.path.exists("output/chandra-braille-ft-new-2/checkpoint-2000") else None
    )

    if trainer.is_fsdp_enabled:
        trainer.accelerator.state.fsdp_plugin.set_state_dict_type("FULL_STATE_DICT")

    trainer.save_model("output/chandra-braille-ft-new-2/final")

    # save processor/tokenizer alongside it
    trainer.processing_class.save_pretrained("output/chandra-braille-ft-new-2/final")
    trainer.save_model(args.output_dir)
    print(f"[Done] model saved to {args.output_dir}")
    processor.save_pretrained(args.output_dir)
    print(f"[Done] saved to {args.output_dir}")


if __name__ == "__main__":
    main()
